# 03. Preparación de los datos

## 1. Objetivo
Limpiar, transformar y estructurar el dataset para dejarlo listo para la etapa de modelado, garantizando consistencia en las variables, tratamiento adecuado de valores inconsistentes, selección de variables útiles y generación de conjuntos de entrenamiento y prueba.

## 2. Carga de datos

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/raw.csv")
df.head()

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,'5-6 hours',Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,'5-6 hours',Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,'Less than 5 hours',Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,'7-8 hours',Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,'5-6 hours',Moderate,M.Tech,Yes,1.0,1.0,No,0


## 3. Limpieza de nombres de columnas

In [2]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_", regex=False)
    .str.replace("?", "", regex=False)
)

df.columns

Index(['id', 'gender', 'age', 'city', 'profession', 'academic_pressure',
       'work_pressure', 'cgpa', 'study_satisfaction', 'job_satisfaction',
       'sleep_duration', 'dietary_habits', 'degree',
       'have_you_ever_had_suicidal_thoughts_', 'work_study_hours',
       'financial_stress', 'family_history_of_mental_illness', 'depression'],
      dtype='object')

## 4. Limpieza de valores inconsistentes

In [3]:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.replace("'", "", regex=False)

In [4]:
df["financial_stress"] = df["financial_stress"].replace("?", np.nan)
df["financial_stress"].value_counts(dropna=False)

financial_stress
5.0    6715
4.0    5775
3.0    5226
1.0    5121
2.0    5061
NaN       3
Name: count, dtype: int64

In [5]:
moda_financial = df["financial_stress"].mode()[0]
df["financial_stress"] = df["financial_stress"].fillna(moda_financial)

## 5. Conversión de tipos de datos

In [6]:
df["financial_stress"] = pd.to_numeric(df["financial_stress"], errors="coerce")

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27901 entries, 0 to 27900
Data columns (total 18 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   id                                    27901 non-null  int64  
 1   gender                                27901 non-null  object 
 2   age                                   27901 non-null  float64
 3   city                                  27901 non-null  object 
 4   profession                            27901 non-null  object 
 5   academic_pressure                     27901 non-null  float64
 6   work_pressure                         27901 non-null  float64
 7   cgpa                                  27901 non-null  float64
 8   study_satisfaction                    27901 non-null  float64
 9   job_satisfaction                      27901 non-null  float64
 10  sleep_duration                        27901 non-null  object 
 11  dietary_habits 

## 6. Estandarización de variables categóricas

In [8]:
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

In [9]:
for col in df.select_dtypes(include="object").columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))


--- gender ---
gender
Male      15547
Female    12354
Name: count, dtype: int64

--- city ---
city
Kalyan           1570
Srinagar         1372
Hyderabad        1340
Vasai-Virar      1290
Lucknow          1155
Thane            1139
Ludhiana         1111
Agra             1094
Surat            1078
Kolkata          1066
Jaipur           1036
Patna            1007
Visakhapatnam     969
Pune              968
Ahmedabad         951
Name: count, dtype: int64

--- profession ---
profession
Student                   27870
Architect                     8
Teacher                       6
Digital Marketer              3
Content Writer                2
Chef                          2
Doctor                        2
Pharmacist                    2
Civil Engineer                1
UX/UI Designer                1
Educational Consultant        1
Manager                       1
Lawyer                        1
Entrepreneur                  1
Name: count, dtype: int64

--- sleep_duration ---
sleep_duration


## 7. Revisión de variables con baja variabilidad

In [10]:
cols_baja_variabilidad = ["work_pressure", "job_satisfaction", "profession"]
df = df.drop(columns=cols_baja_variabilidad)

Se eliminaron las variables work_pressure, job_satisfaction y profession debido a su muy baja variabilidad. En los tres casos, una sola categoría o valor concentra prácticamente la totalidad de los registros, por lo que su aporte discriminante al modelado sería muy limitado.

In [11]:
frecuencia_degree = df["degree"].value_counts()
categorias_frecuentes = frecuencia_degree[frecuencia_degree >= 500].index

df["degree_grouped"] = np.where(df["degree"].isin(categorias_frecuentes), df["degree"], "Other")
df["degree_grouped"].value_counts()

degree_grouped
Class 12    6080
B.Ed        1867
B.Com       1506
B.Arch      1478
BCA         1433
MSc         1190
B.Tech      1152
MCA         1044
M.Tech      1022
BHM          925
Other        893
BSc          888
M.Ed         821
B.Pharm      810
M.Com        734
MBBS         696
BBA          696
LLB          671
BE           613
BA           600
M.Pharm      582
MD           572
MBA          562
MA           544
PhD          522
Name: count, dtype: int64

In [12]:
df = df.drop(columns=["degree"])

In [13]:
frecuencia_city = df["city"].value_counts()
ciudades_frecuentes = frecuencia_city[frecuencia_city >= 500].index

df["city_grouped"] = np.where(df["city"].isin(ciudades_frecuentes), df["city"], "Other")
df = df.drop(columns=["city"])

Aunque se identificaron algunos valores atípicos en variables como age, cgpa y job_satisfaction, su proporción es mínima respecto al total de registros, por lo que en esta etapa no se realizará eliminación de observaciones. Se prioriza conservar la información y evaluar posteriormente si estos valores afectan el desempeño de los modelos.

## 8. Selección de variables finales

In [14]:
df_model = df.copy()
df_model.head()

,id,gender,age,academic_pressure,cgpa,study_satisfaction,sleep_duration,dietary_habits,have_you_ever_had_suicidal_thoughts_,work_study_hours,financial_stress,family_history_of_mental_illness,depression,degree_grouped,city_grouped
0,2,Male,33.0,5.0,8.97,2.0,5-6 hours,Healthy,Yes,3.0,1.0,No,1,B.Pharm,Visakhapatnam
1,8,Female,24.0,2.0,5.90,5.0,5-6 hours,Moderate,No,3.0,2.0,Yes,0,BSc,Bangalore
2,26,Male,31.0,3.0,7.03,5.0,Less than 5 hours,Healthy,No,9.0,1.0,Yes,0,BA,Srinagar
3,30,Female,28.0,3.0,5.59,2.0,7-8 hours,Moderate,Yes,4.0,5.0,Yes,1,BCA,Varanasi
4,32,Female,25.0,4.0,8.13,3.0,5-6 hours,Moderate,Yes,1.0,1.0,No,0,M.Tech,Jaipur


## 9. Separación X e y (Variables predictoras y Objetivo)

In [16]:
X = df_model.drop(columns=["depression", "id"], errors="ignore")
y = df_model["depression"]

## 10. División train/test

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 11. Exportación de datasets procesados

In [19]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

df_model.to_csv("../data/interim/students_depression_clean.csv", index=False)

## 12. Conclusiones